# 3.4.3 Encoding Categorical Variables

One-hot encoding, ordinal encoding, ColumnTransformer, target encoding.

In [ ]:
# Manual OHE
colours=['red','green','blue','red','green']
cats=sorted(set(colours)); ohe={c:i for i,c in enumerate(cats)}
for col in colours:
    vec=[0]*len(cats); vec[ohe[col]]=1; print(f'{col} -> {vec}')

In [ ]:
import numpy as np,pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
np.random.seed(42); n=500
df=pd.DataFrame({'rooms':np.random.randint(1,8,n).astype(float),
                  'age':np.random.randint(1,50,n).astype(float),
                  'zone':np.random.choice(['urban','suburban','rural'],n)})
y=df.rooms*50000+df.age*-1000+df.zone.map({'urban':200000,'suburban':100000,'rural':0})+np.random.normal(0,10000,n)
print(df.head())

In [ ]:
pre=ColumnTransformer([('num',StandardScaler(),['rooms','age']),
                        ('ohe',OneHotEncoder(sparse_output=False),['zone'])])
pipe=Pipeline([('pre',pre),('model',Ridge(1.))])
Xt,Xv,yt,yv=train_test_split(df,y,test_size=.2,random_state=42)
pipe.fit(Xt,yt); print('RMSE:',mean_squared_error(yv,pipe.predict(Xv))**.5:.2f)

In [ ]:
# Target encoding (train-only)
zones=np.random.choice(['urban','suburban','rural'],1000)
y2=np.where(zones=='urban',500000,np.where(zones=='suburban',300000,150000))+np.random.normal(0,20000,1000)
train=np.arange(1000)<800
means={z:y2[train&(zones==z)].mean() for z in ['urban','suburban','rural']}
print('Zone means:',{k:f"{v:.0f}" for k,v in means.items()})